In [1]:
import langchain, langgraph, langsmith, chromadb, pypdf, rank_bm25, flashrank, sentence_transformers
print("All imports OK")

/Users/abdul/Projects/uae-labour-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1022 19:51:27.029000 52861 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


All imports OK


In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)   # override=True: always take the newest values from .env

True

In [3]:
from openai import OpenAI      # already installed as part of langchain-openai
client = OpenAI(base_url=os.environ["NEBIUS_BASE_URL"], api_key=os.environ["NEBIUS_API_KEY"])
for m in client.models.list().data:
    print(m.id)

moonshotai/Kimi-K3
Qwen/Qwen3-235B-A22B-Instruct-2507
google/gemma-3-27b-it
Qwen/Qwen3-Embedding-8B
openai/gpt-oss-120b
Qwen/Qwen3-30B-A3B-Instruct-2507
NousResearch/Hermes-4-405B
nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B
zai-org/GLM-5.1
nvidia/Nemotron-3_5-Lightning
openbmb/MiniCPM-V-4_5
Qwen/Qwen3.8-27B
deepseek-ai/DeepSeek-V4-Flash-0731
MiniMaxAI/MiniMax-M3
moonshotai/Kimi-K2.7-Code
nvidia/Nemotron-3-Ultra-550b-a55b
moonshotai/Kimi-K2.6
nvidia/nemotron-3-super-120b-a12b
Qwen/Qwen3.5-397B-A17B
deepseek-ai/DeepSeek-V4-Pro
deepseek-ai/DeepSeek-V4-Pro-0813
zai-org/GLM-5.2
reflection/beam
nvidia/Nemotron-3.5-Super-VL
meta-models/muse-spark-open
zai-org/GLM-5.3-Flash
deepseek-ai/DeepSeek-V4.1-Flash
zai-org/GLM-5.3


In [3]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())    # finds .env in this folder or any parent folder

def get_llm(temperature=0):
    """Return the LLM chosen in .env, so we can change models without touching code."""
    provider = os.getenv("LLM_PROVIDER", "nebius")
    if provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=os.environ["GROQ_MODEL"], temperature=temperature)
    from langchain_openai import ChatOpenAI           # Nebius uses the OpenAI API format
    return ChatOpenAI(model=os.environ["NEBIUS_MODEL"],
                      base_url=os.environ["NEBIUS_BASE_URL"],
                      api_key=os.environ["NEBIUS_API_KEY"],
                      temperature=temperature)

print("Provider:", os.getenv("LLM_PROVIDER"), "| Tracing:", os.getenv("LANGSMITH_TRACING"))

Provider: nebius | Tracing: true


In [4]:
from openai import OpenAI
model = os.environ["NEBIUS_MODEL"]
print("Model sent:", repr(model))          # repr() shows hidden quotes/spaces

client = OpenAI(base_url=os.environ["NEBIUS_BASE_URL"], api_key=os.environ["NEBIUS_API_KEY"])
ids = [m.id for m in client.models.list().data]
print("Exact match in Nebius list:", model in ids)

Model sent: 'Qwen/Qwen3-30B-A3B-Instruct-2507'
Exact match in Nebius list: True


In [5]:
llm = get_llm()
print(llm.invoke("In one sentence, what is the UAE Labour Law?").content)

The UAE Labour Law is a comprehensive legal framework that governs employment relationships in the United Arab Emirates, outlining the rights and responsibilities of employers and employees, including working hours, leave, termination, and dispute resolution.
